In [ ]:
OPEN_ROUTER_KEY = "OPEN_ROUTER_KEY"
OPEN_API_KEY = "OPEN_API_KEY"

## installation

In [ ]:
!pip install -U \
  "langchain==0.3.*" \
  "langchain-community==0.3.*" \
  "langchain-openai==0.3.*" \
  "langchain-experimental==0.3.4" \
  "sentence-transformers>=3,<4" \
  "faiss-cpu>=1.8"


In [ ]:
!pip install pythainlp

## import

In [ ]:
import os
import re
from pathlib import Path
from typing import List, Dict, Any, Tuple, Optional, Callable, Union
from dataclasses import dataclass
import json

# LangChain imports
from langchain_core.documents import Document
from langchain_experimental.text_splitter import SemanticChunker as LC_SemanticChunker
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.retrievers import BaseRetriever

# Thai NLP
from pythainlp.tokenize import word_tokenize


# Evaluation

## setup llm to judgement

In [ ]:
pip install "ragas>=0.1.11" "langchain-openai>=0.2.2" "datasets>=2.20.0"


In [ ]:
# !pip install -U ragas datasets

In [ ]:
# !pip install --upgrade --force-reinstall pyarrow

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pandas as pd

In [ ]:
!gdown --folder https://drive.google.com/drive/folders/1XXE_TadOwGj6xsYBI7S7Ie5Vpp9HM19o


### calling function

In [ ]:
import json
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness

def evaluate_faithfulness_df(df_in: pd.DataFrame, llm, emb, batch_size: int = 20) -> pd.DataFrame:
    """
    Run faithfulness eval for all records in df_in (batch-friendly).

    Args:
        df_in: DataFrame with ['question','prediction','retrieved_docs']
        llm: ragas_llm (LangchainLLMWrapper)
        emb: ragas embeddings (LangchainEmbeddingsWrapper)
        batch_size: number of rows per batch

    Returns:
        DataFrame with 'faithfulness' score aligned to df_in.index
    """
    n = len(df_in)
    out_chunks = []

    for bi in range(0, n, batch_size):
        chunk = df_in.iloc[bi:bi+batch_size]

        def safe_parse_context(x):
            if isinstance(x, list):
                return x
            if isinstance(x, str):
                x = x.strip()
                # case: looks like JSON list string
                if x.startswith("[") and x.endswith("]"):
                    try:
                        return json.loads(x)
                    except json.JSONDecodeError:
                        pass
                # fallback: wrap entire text in a single-element list
                return [x]
            return [str(x)]

        contexts = chunk["retrieved_docs"].apply(safe_parse_context).tolist()

        ds = Dataset.from_dict({
            "question": chunk["question"].tolist(),
            "answer": chunk["prediction"].tolist(),
            "contexts": contexts,
        })

        sc = evaluate(
            ds,
            metrics=[faithfulness],
            llm=llm,
            embeddings=emb,
        )

        pdf = sc.to_pandas()
        pdf.index = chunk.index
        out_chunks.append(pdf)

    return pd.concat(out_chunks).sort_index()


### get ans

In [ ]:
import pandas as pd


In [ ]:
# retriever_to_use = "s2nd_toc_rewriter_baseline"
# course = ""
# base_path = "/content/chat_ans"
# result_path = "/content/eva"
# df_chatbot_answer = pd.read_csv(f"{base_path}/{retriever_to_use}/answer-{course}-{retriever_to_use}.csv")
df_chatbot_answer = pd.read_csv("/content/qa_results_s2nd_toc_rewriter_baseline.csv")
df_chatbot_answer = df_chatbot_answer.drop_duplicates(subset=['question'], keep='first')
df_chatbot_answer.loc[
    df_chatbot_answer['question'] == 'ใน GIS จำนวนหน่วยกิตตลอดหลักสูตรเป็นเท่าไหร่',
    'reference'
] = 'จำนวนหน่วยกิตตลอดหลักสูตรคือ 127 หน่วยกิตครับ'

df_chatbot_answer

In [ ]:
import shutil
import os

# Define the directory containing the vector database
zip_filename = "eva.zip"
shutil.make_archive(zip_filename.replace(".zip", ""), 'zip', result_path)

print(f"Vector database successfully zipped to: {zip_filename}")

### ans correctness
```
df_ac_summary
```

In [ ]:
# pip install "langchain>=0.2" "tqdm" "pandas" "tenacity"
import os
import re
import asyncio
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import pandas as pd
from tqdm.asyncio import tqdm_asyncio
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate   # ← เพิ่มบรรทัดนี้


# ควบคุมความขนาน (ขึ้นกับ rate limit ของโมเดล)
MAX_CONCURRENCY = 8        # ปรับได้ 4–16 ตามโควต้า/ข้อจำกัด
REQS_PER_SECOND = 3        # ถ้าถูก 429 ให้ลดค่าลง

llm = ChatOpenAI(
    model="x-ai/grok-4-fast",
    temperature=0.0,
    openai_api_key=OPEN_ROUTER_API,
    base_url="https://openrouter.ai/api/v1",
)

answer_correctness_prompt = ChatPromptTemplate.from_template("""
You are an impartial evaluator (LLM-as-a-Judge) for a university curriculum QA system.
You are given three items:

Question: {question}
Model Answer (Prediction): {prediction}
Reference Answer (Ground Truth): {reference}

Your task is to evaluate how correct and aligned the model answer is compared to the reference.
Consider factual accuracy, completeness, and consistency with the reference answer.

Consider factual accuracy, completeness, and consistency with the reference answer.

If the missing answer part is not important, then ignore it.
If the extra information provided is contextually relevant and does not contradict
the reference, it should not lower the score.

Scoring rubric:
0.0 = Incorrect, irrelevant, or hallucinated
0.25  = Partially correct but missing major information
0.5  = Mostly correct with some inaccuracies or omissions
0.75  = Correct and sufficient but not perfectly detailed
1.0  = Fully correct, comprehensive, and well-aligned

Please respond in **two clear lines only**:
Reason: <short justification in English>
Score: <0.0 - 1.0>
""")
# ==== 3) ตัวช่วยพาร์สอย่างทนทาน ====
score_re = re.compile(r"Score\s*:\s*([0-9]+(?:\.[0-9]+)?)", re.I)
reason_re = re.compile(r"Reason\s*:\s*(.*)", re.I)

def parse_reason_score(text: str):
    text = (text or "").strip()
    # หา reason
    reason_match = reason_re.search(text)
    reason = reason_match.group(1).strip() if reason_match else ""
    # หา score
    score_match = score_re.search(text.replace(",", "."))
    score_str = score_match.group(1) if score_match else None
    # ถ้าสกอร์เป็นทศนิยม (0-1) หรือ 1–5 เราจะพยายาม normalize เป็นตัวเลขสตริงเดิม
    return {"score": score_str, "reason": reason or None, "raw": text}

# ==== 4) ฟังก์ชันเรียกโมเดลแบบ async พร้อม retry/backoff ====
@retry(
    reraise=True,
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=20),
    retry=retry_if_exception_type(Exception),
)
async def judge_one(row: pd.Series, sem: asyncio.Semaphore):
    # สร้าง message
    prompt = answer_correctness_prompt.format_messages(
        question=row["question"],
        prediction=row["prediction"],
        reference=row["reference"],
    )
    async with sem:
        # rate limit แบบง่าย: เว้นช่วงตาม REQS_PER_SECOND
        # (ถ้าคุณมีโควต้ามากกว่านี้/ใช้ queue ก็ปรับได้)
        await asyncio.sleep(1.0 / max(1, REQS_PER_SECOND))
        res = await llm.ainvoke(prompt)
    return parse_reason_score(res.content)

# ==== 5) ประเมินแบบขนานด้วย asyncio ====
async def evaluate_parallel(df: pd.DataFrame) -> pd.DataFrame:
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    tasks = [judge_one(row, sem) for _, row in df.iterrows()]

    # ใช้ gather ของ tqdm.asyncio → คงลำดับผลลัพธ์ = ลำดับ tasks เดิม
    ordered_results = await tqdm_asyncio.gather(*tasks, total=len(tasks), desc="Judging")

    out = df.copy()
    out["score"] = [r.get("score") for r in ordered_results]
    out["reason"] = [r.get("reason") for r in ordered_results]
    out["llm_judge_raw"] = [r.get("raw") for r in ordered_results]
    return out



In [ ]:
df_ac = df_chatbot_answer[['question','prediction','reference']]
df_ac = await evaluate_parallel(df_ac)
df_ac[["question", "score", "reason"]]

df_ac_summary = df_ac.copy()
df_ac_summary.rename(columns={'score': 'answer_correctness'}, inplace=True)
df_ac_summary

In [ ]:
df_ac_summary.to_csv('rubric.csv',index=False)

### faithfulness
```
df_ff_summary
```

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_correctness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pandas as pd

# ----- LLM: ใช้ OpenRouter (DeepSeek) -----
llm = ChatOpenAI(
    model="google/gemini-2.5-flash-lite-preview-09-2025",
    temperature=0.0,  # ให้ deterministic ขึ้น
    # ค่านี้จะอ่านจาก env: OPENAI_API_KEY + OPENAI_BASE_URL = OpenRouter
    openai_api_key=OPEN_ROUTER_API,  # picked up automatically if set
    base_url="https://openrouter.ai/api/v1",        # only if using OpenRouter-compatible
)

ragas_llm = LangchainLLMWrapper(llm)
# ----- Embeddings: ใช้ OpenAI แท้ -----
# อย่าให้ OpenAIEmbeddings อ่าน OPENAI_BASE_URL ของ OpenRouter
# เรา “ส่งคีย์ตรง” และ “ไม่ส่ง base_url” เพื่อบังคับให้ไปที่ OpenAI แท้
emb_openai = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPEN_API_KEY,
)


# wrap ให้เป็นอินเทอร์เฟซของ ragas
emb = LangchainEmbeddingsWrapper(emb_openai)

# quick self-check: ต้องได้เวกเตอร์ยาว 1536
test_vec = emb_openai.embed_query("hello world")
print("Embedding dim:", len(test_vec))


In [ ]:
df_ff = df_chatbot_answer[['question','prediction','retrieved_docs']]
df_scores_faith = evaluate_faithfulness_df(df_ff, ragas_llm, emb)

In [ ]:
df_scores_faith.sort_values(by='faithfulness', ascending=False)

In [ ]:
df_scores_faith.fillna(0, inplace=True)

In [ ]:
df_ff_summary = df_scores_faith.copy()
df_ff_summary.rename(columns={'user_input':'question'}, inplace=True)
# df_ff_summary.to_csv(f"{result_path}/{retriever_to_use}/faithfulness-{course}-{retriever_to_use}.csv", index=False, encoding="utf-8-sig")

In [ ]:
df_ff_summary = df_scores_faith.copy()
df_ff_summary.rename(columns={'user_input':'question'}, inplace=True)

In [ ]:
df_ff_summary.info()

In [ ]:
df_ff_summary.to_csv('faithfulness.csv',index=False)

### Context Precision
```
df_ctx_pre_result
```

In [ ]:
df_ctx_pre = df_chatbot_answer[['question','reference','retrieved_docs']]
df_ctx_pre["retrieved_docs"] = df_ctx_pre["retrieved_docs"].apply(lambda x: [x])
df_ctx_pre.head(3)

In [ ]:
import re
import asyncio
from tqdm import tqdm
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithReference
from langchain_openai import ChatOpenAI

llm_ctx_pre= ChatOpenAI(
    model="google/gemini-2.5-flash-lite-preview-09-2025",
    temperature=0.0,            # ให้ determinism สูง
    # max_tokens=512,             # กันตอบยาวเกินจำเป็น
    # timeout=60,                 # กันค้าง
    openai_api_key=OPEN_ROUTER_KEY
    base_url="https://openrouter.ai/api/v1",
    # response_format={"type": "json_object"},
)

context_precision_metric = LLMContextPrecisionWithReference(llm=llm_ctx_pre)

async def compute_llm_context_precision_parallel(df_ctx_pre, evaluator_llm, max_concurrency: int = 3, show_progress=True):
    metric = LLMContextPrecisionWithReference(llm=evaluator_llm)
    regex_pattern = r"__.*?__\n\s*(.*?)(?=\n\n---Context---|$)"

    samples = []
    for _, row in df_ctx_pre.iterrows():
        text = row["retrieved_docs"][0]

        # 🩵 ตัด context ให้เป็น list ก่อน
        if isinstance(text, str):
            matches = re.findall(regex_pattern, text, flags=re.DOTALL)
            retrieved_contexts_clean = [m.strip() for m in matches]
        elif isinstance(text, list):
            retrieved_contexts_clean = text
        else:
            retrieved_contexts_clean = []

        samples.append(
            SingleTurnSample(
                user_input=row["question"],
                reference=row["reference"],
                retrieved_contexts=retrieved_contexts_clean,
            )
        )

    sem = asyncio.Semaphore(max_concurrency)
    results = [None] * len(samples)

    async def evaluate_sample(i, sample):
        async with sem:
            try:
                score = await metric.single_turn_ascore(sample)
                return (i, score)
            except Exception as e:
                print(f"⚠️ Error in row {i}: {e}")
                return (i, None)

    tasks = [evaluate_sample(i, s) for i, s in enumerate(samples)]

    if show_progress:
        for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            i, score = await fut
            results[i] = score
    else:
        results = await asyncio.gather(*tasks)

    df_ctx_pre["ctx_pre_score"] = results
    return df_ctx_pre


In [ ]:
df_ctx_pre_result = await compute_llm_context_precision_parallel(df_ctx_pre, llm_ctx_pre)
df_ctx_pre_result.head()

In [ ]:
df_ctx_pre_result

In [ ]:
df_ctx_pre_result.to_csv('ctx_precision.csv',index=False)

### context recall
```
df_ctx_recall_result
```

In [ ]:
df_ctx_recall = df_chatbot_answer[['question','prediction','reference','retrieved_docs']]
df_ctx_recall["retrieved_docs"] = df_ctx_recall["retrieved_docs"].apply(lambda x: [x])
df_ctx_recall.head(3)

In [ ]:
import re
import asyncio
from tqdm import tqdm
from ragas import SingleTurnSample
from ragas.metrics import LLMContextRecall

from langchain_openai import ChatOpenAI

llm_ctx_recall= ChatOpenAI(
    model="x-ai/grok-4-fast",
    temperature=0.0,
    openai_api_key=OPEN_ROUTER_KEY,
    base_url="https://openrouter.ai/api/v1",
#    response_format={"type": "json_object"},
)

context_recall_metric = LLMContextRecall(llm=llm_ctx_recall)
async def compute_llm_context_recall_parallel(df_ctx_recall, evaluator_llm, max_concurrency: int = 8, show_progress=True):
    metric = LLMContextRecall(llm=evaluator_llm)
    regex_pattern = r"__.*?__\n\s*(.*?)(?=\n\n---Context---|$)"

    samples = []
    for _, row in df_ctx_recall.iterrows():
        text = row["retrieved_docs"][0]

        if isinstance(text, str):
            matches = re.findall(regex_pattern, text, flags=re.DOTALL)
            retrieved_contexts_clean = [m.strip() for m in matches]
        elif isinstance(text, list):
            retrieved_contexts_clean = text
        else:
            retrieved_contexts_clean = []

        samples.append(
            SingleTurnSample(
                user_input=row["question"],
                response=row['prediction'],
                reference=row["reference"],

                retrieved_contexts=retrieved_contexts_clean,
            )
        )

    sem = asyncio.Semaphore(max_concurrency)
    results = [None] * len(samples)

    async def evaluate_sample(i, sample):
        async with sem:
            try:
                score = await metric.single_turn_ascore(sample)
                return (i, score)
            except Exception as e:
                print(f"⚠️ Error in row {i}: {e}")
                return (i, None)

    tasks = [evaluate_sample(i, s) for i, s in enumerate(samples)]

    if show_progress:
        for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
            i, score = await fut
            results[i] = score
    else:
        results = await asyncio.gather(*tasks)

    df_ctx_recall["ctx_recall_score"] = results
    return df_ctx_recall

In [ ]:
df_ctx_recall = await compute_llm_context_recall_parallel(df_ctx_recall, llm_ctx_recall)
df_ctx_recall

In [ ]:
df_ctx_recall.to_csv('ctx_recall.csv',index=False)

# merge all rubric faithfulness ctx_Precision ctx_Recall

In [ ]:
rubric_score = pd.read_csv('/content/rubric.csv')
faithfulness_score = pd.read_csv('/content/faithfulness.csv')
ctx_precision_score = pd.read_csv('/content/ctx_precision.csv')
ctx_recall_score = pd.read_csv('/content/ctx_recall.csv')
ref_ctx = pd.read_csv('/content/reference_ctx.csv')

In [ ]:
rubric_score.head(1)
rubric_score.columns

In [ ]:
faithfulness_score.head(1)
faithfulness_score.columns

In [ ]:
ctx_precision_score.head(1)
ctx_precision_score.columns

In [ ]:
ctx_recall_score.head(1)
ctx_recall_score.columns

In [ ]:
import pandas as pd

# รวมข้อมูลทั้งหมดจาก column "question"
all_process_df = (
    rubric_score
    .merge(
        faithfulness_score[['question', 'faithfulness']],
        on='question',
        how='left'
    )
    .merge(
        ctx_precision_score[['question', 'retrieved_docs', 'ctx_pre_score']],
        on='question',
        how='left'
    )
    .merge(
        ctx_recall_score[['question', 'ctx_recall_score']],
        on='question',
        how='left'
    )
    .merge(
        ref_ctx[['question', 'reference_context']],
        on='question',
        how='left'
    )

)

# เลือกเฉพาะคอลัมน์ที่ต้องการแสดง
all_process_df = all_process_df[
    [
        'question',
        'prediction',
        'reference',
        'answer_correctness',
        'reason',
        'faithfulness',
        'retrieved_docs',
        'ctx_pre_score',
        'ctx_recall_score',
        'reference_context'

    ]
]

# แสดงผลลัพธ์
all_process_df.to_csv('all_process.csv', index=False)


In [ ]:
all_process_df